In [ ]:
# Import necessary libraries
import json
import anthropic
import concurrent.futures

In [ ]:
# Import environment variables from .env file
from dotenv import load_dotenv 
load_dotenv()

In [ ]:

def add_user_messages(messages, message):
    messages.append({"role": "user", "content": message})

def add_assistant_messages(messages, message):
    messages.append({"role": "assistant", "content": message})
    
def chat(messages, model="claude-haiku-4-5", system=None, temperature=1.0, stop_sequences=[]):
    parameters = {
        "model": model,
        "max_tokens": 1024,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }
    
    if system:
        parameters["system"] = system
    
    client = anthropic.Anthropic()
    response = client.messages.create(**parameters)
    return response.content[0].text

In [ ]:
def run_prompt(prompt_inputs):
    print(f"Running prompt for task: with inputs: {prompt_inputs}\n")
    prompt = f"""
        Create a personalized meal plan for a athlete with the following details:
        <athlete_details>
        - Height: {prompt_inputs["height"]}
        - Weight: {prompt_inputs["weight"]}
        - Goal: {prompt_inputs["goal"]}
        - Dietary restrictions: {prompt_inputs["restrictions"]}
        </athlete_details>

        Guidelines:
        1. Include accurate daily calorie amount
        2. Show protein, fat, and carb amounts  
        3. Specify when to eat each meal
        4. Use only foods that fit restrictions
        5. List all portion sizes in grams
        6. Keep budget-friendly if mentioned

        Here is an example with a sample input and an ideal output:
        <specs>        
        - "height": "165",
        - "weight": "58",
        - "goal": "Weight loss and endurance improvement",
        - "restrictions": "Vegetarian, no dairy"
        </specs>

        <ideal_output>
        Here is a one-day meal plan for an athlete aiming to maintain fitness and improve cholesterol levels:

        *   **Calorie Target:** Approximately 2500 calories
        *   **Macronutrient Breakdown:** Protein (140g), Fat (70g), Carbs (340g)

        **Meal Plan:**

        *   **Breakfast (7:00 AM):** Oatmeal (80g dry weight) with berries (100g) and walnuts (15g). Skim milk (240g).
            *   Protein: 15g, Fat: 15g, Carbs: 60g
        *   **Mid-Morning Snack (10:00 AM):** Apple (150g) with almond butter (30g).
            *   Protein: 7g, Fat: 18g, Carbs: 25g
        *   **Lunch (1:00 PM):** Grilled chicken breast (120g) salad with mixed greens (150g), cucumber (50g), tomato (50g), and a light vinaigrette dressing (30g). Whole wheat bread (60g).
            *   Protein: 40g, Fat: 15g, Carbs: 70g
        *   **Afternoon Snack (4:00 PM):** Greek yogurt (170g, non-fat) with a banana (120g).
            *   Protein: 20g, Fat: 0g, Carbs: 40g
        *   **Dinner (7:00 PM):** Baked salmon (140g) with steamed broccoli (200g) and quinoa (75g dry weight).
            *   Protein: 40g, Fat: 20g, Carbs: 80g
        *   **Evening Snack (9:00 PM):** Small handful of almonds (20g).
            *   Protein: 8g, Fat: 12g, Carbs: 15g

        This meal plan prioritizes lean protein sources, whole grains, fruits, and vegetables, while limiting saturated and trans fats to support healthy cholesterol levels.
        </ideal_output>

        """
    messages = []
    add_user_messages(messages, prompt)
    return {
        "output": chat(messages),
        "task": prompt_inputs
    }

def grade_by_model(solution_criteria, output, specs):
    prompt = f"""
        Overview output and assess content to meeting on solution criteria based on specs.
        Output:
        <output>
        {output}
        </output>
        Criteria:
        <criteria>
        {solution_criteria}
        </criteria>
        Specs:
        <specs>
        {specs}
        </specs>

        Provide your evaluation as JSON object:
        - grade: a decimal number with two decimal places from 0.00 to 10.00, where 10.00 is the best grade.
        - comment: short explanation of the grade.

        # Restrictions:
        - Output should be without any markdown or code block, only JSON object. 
        - Don't wrap output in any markdown. Don't use any backticks.
        """
    messages = []
    add_user_messages(messages, prompt)
    grade = chat(messages, system="""
                 Output should be without any markdown or code block, only JSON object. 
                 Don't wrap output in any markdown. Don't use any backticks.
                 """)
    print(grade)
    return json.loads(grade)

class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=1):
        self.max_concurrent_tasks = max_concurrent_tasks

    def generate_dataset(
        self,
        task,
        prompt_specs={},
        output_file="dataset-prompt-techniques.json",
        num_cases=3
    ):
        messages = []
        add_user_messages(
            messages, 
            f"""
            Generate a dataset of unique {num_cases} elements for task:
            <task> 
            {task}
            </task> 
            
            Output has to include the below specification:
            <specification>
            {prompt_specs}
            </specification>

            Include into output a solution criteria field named `solution_criteria` for each case. 
            It should be a key point that defines whether the solution is correct or not. 

            # Restrictions: 
            - The output should be a JSON an array of strings.
            - Don't wrap it in any markdown or code block.
            - Don't use any backticks in output.
            """
            )
        response = chat(messages, system="Output should be a JSON array of strings. Don't wrap it in any markdown or code block. Don't use any backticks in output.")
        print(response)
        with open(output_file, "w") as f:
            json.dump(json.loads(response), f, indent=4)
    
    def run_evaluation(self, dataset):
        with open(dataset, "r") as f:
            tasks = json.load(f)

        results = []
        # for task in tasks: 
        #     output = run_prompt(task)
        #     results.append({
        #         "task": task,
        #         "output": output,
        #         "grade_by_model": grade_by_model(task["solution_criteria"], output, tasks),
        #     })
            
        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            futures = {
                executor.submit(
                    run_prompt,
                    task
                ): task
                for task in tasks
            }

            for future in concurrent.futures.as_completed(futures):
                result = future.result()
                results.append({
                    "task": result["task"],
                    "output": result["output"],
                    "grade_by_model": grade_by_model(result["task"]["solution_criteria"], result["output"], result["task"]),
                })

        print(results)
        with open("evaluation_results.json", "w") as f:
            json.dump(results, f, indent=4)


In [ ]:
evaluator = PromptEvaluator(max_concurrent_tasks=3)
# evaluator.generate_dataset(
#     task="Write compact, consice 1 day meal plan for a single athlete",
#     prompt_specs="""
#     {
#         "height": "Athlete's height in cm",
#         "weight": "Athlete's weight in kg",
#         "goal": "Goal of the athlete",
#         "restrictions": "Dietary restrictions of the athlete"
#     }
#     """,
# )

evaluator.run_evaluation("dataset-prompt-techniques.json")